In [0]:
# %sh databricks secrets create-scope joshua_scope

In [0]:
# from databricks.sdk import WorkspaceClient

# w = WorkspaceClient()

# # Create a Databricks-backed scope for yourself
# scope_name = "joshua_scope"

# try:
#     w.secrets.create_scope(scope=scope_name)
#     print(f"✅ Scope '{scope_name}' created successfully!")
# except Exception as e:
#     print(f"ℹ️ Scope creation note: {e}")

In [0]:
# List all secret scopes in the workspace
display(dbutils.secrets.listScopes())

In [0]:
display(dbutils.secrets.list("joshua_scope"))

In [0]:
# import time
# import json
# import requests
# import datetime
# from azure.eventhub import EventHubProducerClient, EventData

# # Configuration
# SECRET_SCOPE = "joshua_scope"
# SECRET_KEY = "joshuandegwa-evh-cs"
# EVENT_HUB_NAME = "JoshuaNdegwa-evh"  # Update if your event hub name is different

# OPENSKY_URL = "https://opensky-network.org/api/states/all?lamin=-4.5&lomin=33.9&lamax=5.0&lomax=41.9"

# # Retrieve connection string from secret
# connection_string = dbutils.secrets.get(SECRET_SCOPE, SECRET_KEY)

# # Fetch OpenSky data
# headers = {"User-Agent": "DatabricksFlightProducer/1.0 (joshua.ndegwa@fmcg.com)"}
# response = requests.get(OPENSKY_URL, headers=headers, timeout=10)

# if response.status_code == 200:
#     payload = response.json()
#     raw_states = payload.get("states") or []
#     api_time = payload.get("time")
    
#     parsed_flights = []
#     for state in raw_states:
#         parsed_flights.append({
#             "icao24": state[0],
#             "callsign": state[1].strip() if state[1] else "UNKNOWN",
#             "origin_country": state[2],
#             "longitude": state[5],
#             "latitude": state[6],
#             "baro_altitude_m": state[7],
#             "on_ground": state[8],
#             "velocity_mps": state[9],
#             "true_track_deg": state[10],
#             "api_timestamp": api_time,
#             "ingested_at": datetime.datetime.now(datetime.timezone.utc).isoformat()
#         })

#     # Send to Event Hub
#     producer = EventHubProducerClient.from_connection_string(
#         conn_str=connection_string,
#         eventhub_name=EVENT_HUB_NAME
#     )
    
#     with producer:
#         batch = producer.create_batch()
#         for flight in parsed_flights:
#             try:
#                 batch.add(EventData(json.dumps(flight)))
#             except ValueError:
#                 producer.send_batch(batch)
#                 batch = producer.create_batch()
#                 batch.add(EventData(json.dumps(flight)))
#         if len(batch) > 0:
#             producer.send_batch(batch)
            
#     print(f"🚀 Successfully published {len(parsed_flights)} flight events to Event Hub '{EVENT_HUB_NAME}'!")
# else:
#     print(f"⚠️ OpenSky API failed with status code {response.status_code}")

In [0]:
import json
import time
import random
from azure.eventhub import EventHubProducerClient, EventData

# -------------------------------------------------------------------
# 1. Configuration & Secrets Retrieval
# -------------------------------------------------------------------
SECRET_SCOPE = "joshua_scope"
SECRET_KEY = "joshuandegwa-evh-cs"
EVENT_HUB_NAME = "JoshuaNdegwa-evh"

connection_string = dbutils.secrets.get(scope=SECRET_SCOPE, key=SECRET_KEY)

# -------------------------------------------------------------------
# 2. Initialize Event Hub Producer Client
# -------------------------------------------------------------------
producer = EventHubProducerClient.from_connection_string(
    conn_str=connection_string,
    eventhub_name=EVENT_HUB_NAME
)

# Sample countries and callsigns for simulation
countries = ["United States", "United Kingdom", "Germany", "France", "Kenya", "Canada", "Japan"]
airlines = ["KL", "BA", "AF", "KQ", "LH", "UA", "DL"]

print("🚀 Starting high-volume flight data producer...")

# -------------------------------------------------------------------
# 3. Generate and Send Batches of Flight Records
# -------------------------------------------------------------------
BATCH_COUNT = 5  # Number of batches to send
MESSAGES_PER_BATCH = 50

try:
    with producer:
        for batch_idx in range(BATCH_COUNT):
            event_data_batch = producer.create_batch()
            
            for i in range(MESSAGES_PER_BATCH):
                flight_payload = {
                    "icao24": f"a{random.randint(100000, 999999):x}",
                    "callsign": f"{random.choice(airlines)}{random.randint(100, 999)}",
                    "origin_country": random.choice(countries),
                    "longitude": round(random.uniform(-180.0, 180.0), 4),
                    "latitude": round(random.uniform(-90.0, 90.0), 4),
                    "baro_altitude_m": round(random.uniform(500.0, 12000.0), 2),
                    "on_ground": False,
                    "velocity_mps": round(random.uniform(150.0, 300.0), 2),
                    "true_track_deg": round(random.uniform(0.0, 360.0), 2),
                    "api_timestamp": int(time.time()),
                    "ingested_at": time.strftime("%Y-%m-%d %H:%M:%S")
                }
                
                # Add to event batch
                event_data_batch.add(EventData(json.dumps(flight_payload)))
            
            # Send the batch to Event Hubs
            producer.send_batch(event_data_batch)
            print(f"✅ Sent batch {batch_idx + 1}/{BATCH_COUNT} ({MESSAGES_PER_BATCH} messages)")
            print(flight_payload)
            time.sleep(1) # Brief pause between batches

    print("🎉 Successfully finished pushing flight data to Event Hub!")

except Exception as e:
    print(f"❌ Error sending events: {e}")